# Multistage mission setup example

This notebook walks through a minimal, step-by-step setup of the multistage
mission objects (`BodyLike`, `Stage`, `Deployable`, `Mission`). It also
runs simplified flight simulations for Stage 1, Stage 2, and the payload
using the mission Stage/Deployable bodies as the source of each flight
model so the mission architecture remains the single source of truth
while full coupling continues to evolve.


In [1]:
from rocketpy import (
    Attachment,
    Deployable,
    DeploymentEvent,
    Environment,
    Flight,
    FlightBody,
    IgnitionEvent,
    Mission,
    PointMassMotor,
    Rocket,
    Stage,
    StageSeparationEvent,
)

## 1. Define simple time-invariant models

In [2]:
def constant_mass(value):
    def model(t):
        return float(value)

    return model


def constant_inertia_tensor(i11, i22, i33):
    def model(t):
        return (i11, i22, i33, 0.0, 0.0, 0.0)

    return model


def constant_center_of_mass(value):
    def model(t):
        return float(value)

    return model

## 2. Create the root body (Stage 1)

In [3]:
stage1_body = FlightBody(
    name="Stage 1",
    geometry={"radius": 0.08, "length": 2.4},
    mass_model=constant_mass(18.0),
    inertia_model=constant_inertia_tensor(2.8, 2.8, 0.15),
    center_of_mass_model=constant_center_of_mass(1.1),
)

print("Stage 1:", stage1_body.name)
print("  mass @ t=0 s:", stage1_body.mass(0.0), "kg")
print("  center of mass @ t=0 s:", stage1_body.center_of_mass(0.0), "m")

Stage 1: Stage 1
  mass @ t=0 s: 18.0 kg
  center of mass @ t=0 s: 1.1 m


## 3. Create the upper-stage body (Stage 2)

In [4]:
stage2_body = FlightBody(
    name="Stage 2",
    geometry={"radius": 0.06, "length": 1.4},
    mass_model=constant_mass(7.5),
    inertia_model=constant_inertia_tensor(1.1, 1.1, 0.06),
    center_of_mass_model=constant_center_of_mass(0.7),
)

print("Stage 2:", stage2_body.name)
print("  mass @ t=0 s:", stage2_body.mass(0.0), "kg")

Stage 2: Stage 2
  mass @ t=0 s: 7.5 kg


## 4. Define attachment and events for Stage 2

In [ ]:
stage2_attachment = Attachment(
    parent_frame_position=[0.0, 0.0, 2.2],
    child_frame_position=[0.0, 0.0, 0.0],
    constraints="rigid",
)

STAGE2_IGNITION_TIME = 2.0
STAGE2_SEPARATION_TIME = 5.0


def ignition_trigger(state, context):
    return context["time"] >= STAGE2_IGNITION_TIME


def separation_trigger(state, context):
    return context["time"] >= STAGE2_SEPARATION_TIME


ignition_event = IgnitionEvent("stage2_ignite", ignition_trigger)
separation_event = StageSeparationEvent("stage2_separate", separation_trigger)

context = {"time": 3.0}
print("Ignition fires @ t=3 s?", ignition_event.should_fire({}, context))
print("Separation fires @ t=3 s?", separation_event.should_fire({}, context))

## 5. Wrap Stage 2 as a mission `Stage`

In [6]:
stage2 = Stage(
    name="Stage 2",
    body=stage2_body,
    attachment=stage2_attachment,
    separation_event=separation_event,
    ignition_event=ignition_event,
)

print("Stage object:", stage2)
print("  state:", stage2.state)
print("  events:", [event.name for event in stage2.events])

Stage object: Stage(name='Stage 2')
  state: StageState.ATTACHED
  events: ['stage2_separate', 'stage2_ignite']


## 6. Create a deployable payload

In [ ]:
payload_body = FlightBody(
    name="Payload",
    geometry={"radius": 0.05, "length": 0.4},
    mass_model=constant_mass(1.2),
    inertia_model=constant_inertia_tensor(0.2, 0.2, 0.01),
    center_of_mass_model=constant_center_of_mass(0.2),
)

payload_attachment = Attachment(
    parent_frame_position=[0.0, 0.0, 2.6],
    child_frame_position=[0.0, 0.0, 0.0],
    constraints="rigid",
)

PAYLOAD_RELEASE_TIME = 8.0


def payload_trigger(state, context):
    return context["time"] >= PAYLOAD_RELEASE_TIME


payload_event = DeploymentEvent("payload_release", payload_trigger)

payload = Deployable(
    name="Payload",
    body=payload_body,
    attachment=payload_attachment,
    deployment_event=payload_event,
)

context = {"time": 9.0}
print("Payload event fires @ t=9 s?", payload_event.should_fire({}, context))

## 7. Build the Mission

In [8]:
mission = Mission(name="Two-Stage Demo")
mission.add_stage(stage2)
mission.add_deployable(payload)

print(mission)
print("Attached items:", [item.name for item in mission.attached_items()])

Mission(name='Two-Stage Demo', stages=1, deployables=1)
Attached items: ['Stage 2', 'Payload']


## 8. Run simplified flight simulations

Run a simplified flight for Stage 1, then use the Stage 2 and payload
bodies from the mission to configure the upper-stage and deployable
trajectories. The Stage 1 trajectory provides the initial conditions at
each separation event.


In [ ]:
environment = Environment(latitude=0.0, longitude=0.0, elevation=0.0)

stage1_motor = PointMassMotor(
    thrust_source=480.0,
    dry_mass=1.2,
    propellant_initial_mass=2.8,
    burn_time=2.4,
)

stage1_rocket = Rocket(
    radius=stage1_body.geometry["radius"],
    mass=stage1_body.mass(0.0),
    inertia=stage1_body.inertia_tensor(0.0),
    power_off_drag=0.45,
    power_on_drag=0.45,
    center_of_mass_without_motor=stage1_body.center_of_mass(0.0),
)
stage1_rocket.add_motor(stage1_motor, position=0.0)

stage1_flight = Flight(
    rocket=stage1_rocket,
    environment=environment,
    rail_length=5.0,
    inclination=85.0,
    heading=0.0,
    max_time=20.0,
    terminate_on_apogee=False,
)

stage2_motor = PointMassMotor(
    thrust_source=260.0,
    dry_mass=0.6,
    propellant_initial_mass=1.2,
    burn_time=1.6,
)

stage2_body = stage2.body
stage2_rocket = Rocket(
    radius=stage2_body.geometry["radius"],
    mass=stage2_body.mass(0.0),
    inertia=stage2_body.inertia_tensor(0.0),
    power_off_drag=0.35,
    power_on_drag=0.35,
    center_of_mass_without_motor=stage2_body.center_of_mass(0.0),
)
stage2_rocket.add_motor(stage2_motor, position=0.0)

stage2_initial_state = stage1_flight.get_solution_at_time(STAGE2_SEPARATION_TIME)

stage2_flight = Flight(
    rocket=stage2_rocket,
    environment=environment,
    rail_length=1.0,
    initial_solution=stage2_initial_state,
    max_time=20.0,
    terminate_on_apogee=False,
)

payload_body = payload.body
payload_rocket = Rocket(
    radius=payload_body.geometry["radius"],
    mass=payload_body.mass(0.0),
    inertia=payload_body.inertia_tensor(0.0),
    power_off_drag=0.3,
    power_on_drag=0.3,
    center_of_mass_without_motor=payload_body.center_of_mass(0.0),
)

payload_initial_state = stage2_flight.get_solution_at_time(PAYLOAD_RELEASE_TIME)

payload_flight = Flight(
    rocket=payload_rocket,
    environment=environment,
    rail_length=1.0,
    initial_solution=payload_initial_state,
    max_time=20.0,
    terminate_on_apogee=False,
)

print(f"Stage 1 apogee: {stage1_flight.apogee:.1f} m")
print(f"Stage 2 apogee: {stage2_flight.apogee:.1f} m")
print(f"Payload apogee: {payload_flight.apogee:.1f} m")